# Laboratorio 04 — Fine-tuning LoRA sobre Llama 3.2 con dataset propio
### Python AI Developer 2026 · Capítulo 2: Prompt Engineering & Fine-tuning

Objetivo: aplicar fine-tuning supervisado (SFT) con LoRA a un modelo pequeño usando un dataset creado por nosotros.

Aprenderemos a entender qué pasa durante el fine-tuning, no solo a ejecutar el código.

**Requisito importante:** este laboratorio requiere GPU. Ejecutarlo en Google Colab con runtime T4 (gratuito).

Duración estimada: 2 horas  
Modelo: Llama 3.2 1B Instruct (más manejable que 8B en Colab T4)

---
## Parte 1 — Instalar dependencias

En Colab, instalar directamente con pip. En local con uv si tienes GPU disponible.

In [ ]:
# Instalar las librerías necesarias
# transformers: para cargar y trabajar con modelos HuggingFace
# peft: para aplicar LoRA (Parameter-Efficient Fine-Tuning)
# trl: SFTTrainer para fine-tuning supervisado
# datasets: para manejar nuestro dataset
# accelerate: para optimizar el entrenamiento
# bitsandbytes: para cuantización (reduce uso de memoria GPU)

# Descomenta y ejecuta si estás en Colab:
# !pip install transformers peft trl datasets accelerate bitsandbytes -q

In [1]:
!pip install -q --upgrade pyarrow
!pip install -q -U trl transformers peft datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


In [2]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# verificar que tenemos GPU disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Sin GPU — el entrenamiento será muy lento. Usar Google Colab.")

Dispositivo: cuda
GPU: Tesla T4
Memoria disponible: 15.6 GB


---
## Parte 2 — Dataset propio

Un buen dataset de fine-tuning tiene:
- Pares instrucción/respuesta consistentes con el estilo deseado
- Suficiente diversidad para generalizar (al menos 50-100 ejemplos para casos simples)
- Calidad sobre cantidad: mejor 50 ejemplos perfectos que 500 mediocres

Nuestro dominio: asistente técnico para desarrolladores Python que responde siempre en un formato estructurado específico.

In [3]:
# Dataset de fine-tuning: enseñamos al modelo a responder preguntas de Python
# con un formato estructurado específico:
# [Concepto] → [Explicación breve] → [Ejemplo de código] → [Cuándo usarlo]
#
# Este formato NO es el comportamiento por defecto del modelo base,
# por eso es un buen caso de fine-tuning: queremos cambiar el estilo de respuesta.

SYSTEM = "Eres un asistente técnico de Python. Responde SIEMPRE con este formato exacto:\n[Concepto]: definición breve\n[Ejemplo]: código funcional\n[Cuándo usarlo]: casos de uso en una oración"

datos_raw = [
    {
        "instruccion": "¿Qué es una list comprehension en Python?",
        "respuesta": "[Concepto]: Sintaxis compacta para crear listas aplicando una expresión a cada elemento de un iterable.\n[Ejemplo]:\n```python\ncuadrados = [x**2 for x in range(10)]\n```\n[Cuándo usarlo]: Cuando necesitas transformar o filtrar una lista en una sola línea legible."
    },
    {
        "instruccion": "¿Qué es un decorador en Python?",
        "respuesta": "[Concepto]: Función que recibe otra función como argumento y extiende su comportamiento sin modificar su código.\n[Ejemplo]:\n```python\ndef log(func):\n    def wrapper(*args):\n        print(f'Llamando {func.__name__}')\n        return func(*args)\n    return wrapper\n\n@log\ndef suma(a, b): return a + b\n```\n[Cuándo usarlo]: Para agregar logging, autenticación o caché a funciones existentes sin modificarlas."
    },
    {
        "instruccion": "¿Qué es un generador en Python?",
        "respuesta": "[Concepto]: Función que produce valores uno a uno usando yield, sin guardar todos en memoria.\n[Ejemplo]:\n```python\ndef contar(n):\n    for i in range(n):\n        yield i\n\nfor num in contar(1000000):  # no ocupa 1M de memoria\n    pass\n```\n[Cuándo usarlo]: Cuando trabajas con secuencias grandes o infinitas donde cargar todo en memoria no es viable."
    },
    {
        "instruccion": "¿Qué es el context manager en Python?",
        "respuesta": "[Concepto]: Protocolo que garantiza la ejecución de código de limpieza (cerrar archivos, liberar recursos) aunque ocurra una excepción.\n[Ejemplo]:\n```python\nwith open('archivo.txt', 'r') as f:\n    contenido = f.read()\n# el archivo se cierra automáticamente aquí\n```\n[Cuándo usarlo]: Siempre que abras archivos, conexiones de BD o locks que deben liberarse."
    },
    {
        "instruccion": "¿Qué es el unpacking en Python?",
        "respuesta": "[Concepto]: Asignación simultánea de múltiples valores de un iterable a variables individuales.\n[Ejemplo]:\n```python\na, b, *resto = [1, 2, 3, 4, 5]\n# a=1, b=2, resto=[3, 4, 5]\n\nprimero, *_, ultimo = range(10)\n```\n[Cuándo usarlo]: Para extraer elementos de listas, tuplas o resultados de funciones que devuelven múltiples valores."
    },
    {
        "instruccion": "¿Qué es el walrus operator en Python?",
        "respuesta": "[Concepto]: Operador := que asigna y retorna un valor en la misma expresión (Python 3.8+).\n[Ejemplo]:\n```python\ndatos = [1, 2, 3, 4, 5]\nif (n := len(datos)) > 3:\n    print(f'Lista con {n} elementos')\n```\n[Cuándo usarlo]: Para evitar calcular dos veces el mismo valor en condiciones o while loops."
    },
    {
        "instruccion": "¿Qué es dataclass en Python?",
        "respuesta": "[Concepto]: Decorador que genera automáticamente métodos __init__, __repr__ y __eq__ para clases que son principalmente contenedores de datos.\n[Ejemplo]:\n```python\nfrom dataclasses import dataclass\n\n@dataclass\nclass Punto:\n    x: float\n    y: float\n    z: float = 0.0\n\np = Punto(1.0, 2.0)\n```\n[Cuándo usarlo]: Para modelar datos estructurados sin escribir boilerplate de __init__ manualmente."
    },
    {
        "instruccion": "¿Qué es functools.lru_cache en Python?",
        "respuesta": "[Concepto]: Decorador que memoriza los resultados de una función para evitar recalcularlos con los mismos argumentos (memoización).\n[Ejemplo]:\n```python\nfrom functools import lru_cache\n\n@lru_cache(maxsize=128)\ndef fibonacci(n):\n    if n < 2: return n\n    return fibonacci(n-1) + fibonacci(n-2)\n```\n[Cuándo usarlo]: Para funciones puras computacionalmente costosas que se llaman repetidamente con los mismos argumentos."
    },
    {
        "instruccion": "¿Qué es asyncio en Python?",
        "respuesta": "[Concepto]: Librería para programación asíncrona que permite ejecutar operaciones concurrentes sin usar múltiples threads.\n[Ejemplo]:\n```python\nimport asyncio\n\nasync def fetch(url):\n    await asyncio.sleep(1)  # simula llamada HTTP\n    return f'datos de {url}'\n\nasync def main():\n    resultados = await asyncio.gather(fetch('a'), fetch('b'))\n    print(resultados)\n```\n[Cuándo usarlo]: Para I/O intensivo como llamadas HTTP, lectura de archivos o consultas a BD donde esperar bloqueado es ineficiente."
    },
    {
        "instruccion": "¿Qué son las f-strings en Python?",
        "respuesta": "[Concepto]: Cadenas literales que permiten insertar expresiones Python directamente dentro de llaves {}.\n[Ejemplo]:\n```python\nnombre = 'Ana'\nprecio = 1234.5678\nprint(f'Hola {nombre}, el precio es ${precio:.2f}')\n# Hola Ana, el precio es $1234.57\n```\n[Cuándo usarlo]: Siempre que necesites construir strings con valores variables; es más legible y rápido que format() o concatenación."
    },
]

print(f"Dataset creado: {len(datos_raw)} ejemplos")

Dataset creado: 10 ejemplos


In [4]:
# Formatear el dataset al esquema de chat que espera el modelo
# El formato ChatML es el estándar para modelos de instrucción:
# <|system|>...<|user|>...<|assistant|>...
#
# Usamos apply_chat_template del tokenizador para que el formato
# sea exactamente el que espera Llama 3.2

def formatear_para_chat(ejemplo):
    """Convierte un par instrucción/respuesta al formato de mensajes del modelo."""
    mensajes = [
        {"role": "system",    "content": SYSTEM},
        {"role": "user",      "content": ejemplo["instruccion"]},
        {"role": "assistant", "content": ejemplo["respuesta"]},
    ]
    return {"messages": mensajes}


datos_formateados = [formatear_para_chat(d) for d in datos_raw]
dataset = Dataset.from_list(datos_formateados)

print(f"Dataset formateado: {len(dataset)} ejemplos")
print("Estructura de un ejemplo:")
print(f"  roles: {[m['role'] for m in dataset[0]['messages']]}")
print(f"  primer mensaje system: {dataset[0]['messages'][0]['content'][:60]}...")

Dataset formateado: 10 ejemplos
Estructura de un ejemplo:
  roles: ['system', 'user', 'assistant']
  primer mensaje system: Eres un asistente técnico de Python. Responde SIEMPRE con es...


---
## Parte 3 — Cargar el modelo base y el tokenizador

Usamos Llama 3.2 1B Instruct. El '1B' indica ~1 mil millones de parámetros.
Es suficientemente pequeño para Colab T4 y suficientemente capaz para esta tarea.

In [5]:
from dotenv import load_dotenv

load_dotenv()

# Verificar que las tres keys están presentes (sin mostrar sus valores)
keys = {
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY"),
    "HF_TOKEN": os.getenv("HF_TOKEN"),
}

In [20]:
# Necesitas haber aceptado los términos de Llama en HuggingFace https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
# y haber configurado tu token HF_TOKEN en Colab secrets o .env

# Si estás en Colab, autenticarte así:
from huggingface_hub import login
login(token=keys["HF_TOKEN"])

NOMBRE_MODELO = "meta-llama/Llama-3.2-1B-Instruct"
#NOMBRE_MODELO = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Cargando tokenizador: {NOMBRE_MODELO}")
tokenizador = AutoTokenizer.from_pretrained(NOMBRE_MODELO)

# padding_side='right' es necesario para SFTTrainer
tokenizador.padding_side = "right"

print(f"Vocabulario del tokenizador: {tokenizador.vocab_size} tokens")
print(f"Token de padding: {tokenizador.pad_token}")
print(f"Token de fin de secuencia: {tokenizador.eos_token}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cargando tokenizador: meta-llama/Llama-3.2-1B-Instruct


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Vocabulario del tokenizador: 128000 tokens
Token de padding: None
Token de fin de secuencia: <|eot_id|>


In [21]:
# Cargar el modelo con cuantización de 4 bits para reducir uso de memoria
# Sin cuantización, Llama 3.2 1B ocupa ~2GB en float32
# Con cuantización 4 bits, baja a ~500MB
# Sin cuantización, Qwen2.5 0.5B ocupa ~1GB en float32
# Con cuantización 4 bits, baja a ~250MB

from transformers import BitsAndBytesConfig

config_cuantizacion = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # doble cuantización ahorra un poco más
    bnb_4bit_quant_type="nf4",       # NF4 mantiene mejor precisión que int4 puro
)

print(f"Cargando modelo: {NOMBRE_MODELO}")
modelo = AutoModelForCausalLM.from_pretrained(
    NOMBRE_MODELO,
    quantization_config=config_cuantizacion,
    device_map="auto",  # asigna capas automáticamente a GPU/CPU
)

# contar parámetros totales del modelo base
total_params = sum(p.numel() for p in modelo.parameters())
print(f"Parámetros totales del modelo: {total_params:,}")

Cargando modelo: meta-llama/Llama-3.2-1B-Instruct


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Parámetros totales del modelo: 749,275,136


---
## Parte 4 — Configurar LoRA

LoRA congela todos los pesos del modelo original y agrega matrices pequeñas (adaptadores)
en capas específicas. Solo entrenamos esas matrices pequeñas.

Los parámetros clave:
- r: rango de las matrices LoRA. Más alto = más capacidad pero más parámetros
- lora_alpha: factor de escala. Típicamente 2*r o igual a r
- target_modules: en qué capas del modelo insertar los adaptadores

In [22]:
config_lora = LoraConfig(
    r=16,                    # rango: balance entre capacidad y eficiencia
    lora_alpha=32,           # escala: típicamente 2*r
    target_modules=[         # capas de atención donde insertar los adaptadores
        "q_proj",            # query projection
        "k_proj",            # key projection
        "v_proj",            # value projection
        "o_proj",            # output projection
    ],
    lora_dropout=0.05,       # dropout para regularización
    bias="none",             # no entrenar los bias
    task_type=TaskType.CAUSAL_LM,  # tipo de tarea: generación de texto
)

modelo = get_peft_model(modelo, config_lora)

# ver cuántos parámetros son entrenables vs congelados
params_entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
params_totales = sum(p.numel() for p in modelo.parameters())

print(f"Parámetros entrenables: {params_entrenables:,}")
print(f"Parámetros totales:     {params_totales:,}")
print(f"Porcentaje entrenable:  {100 * params_entrenables / params_totales:.2f}%")
print()
print("Con LoRA solo entrenamos una fracción pequeña de los parámetros.")
print("Eso es exactamente el punto: misma calidad, mucho menos cómputo.")

Parámetros entrenables: 3,407,872
Parámetros totales:     752,683,008
Porcentaje entrenable:  0.45%

Con LoRA solo entrenamos una fracción pequeña de los parámetros.
Eso es exactamente el punto: misma calidad, mucho menos cómputo.


---
## Parte 5 — Inferencia antes del fine-tuning (línea base)

Siempre testear el modelo base ANTES de entrenar para tener un punto de comparación.

In [30]:
def generar_respuesta(pregunta: str, modelo_actual, tokenizador_actual) -> str:
    # poner el modelo en modo evaluación — importante después del entrenamiento
    modelo_actual.eval()

    mensajes = [
        {"role": "system",  "content": SYSTEM},
        {"role": "user",    "content": pregunta},
    ]
    prompt = tokenizador_actual.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizador_actual(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = modelo_actual.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            do_sample=True,
            repetition_penalty=1.3,   # penaliza repetir el mismo token
            pad_token_id=tokenizador_actual.eos_token_id,
        )

    tokens_nuevos = output[0][inputs["input_ids"].shape[1]:]
    return tokenizador_actual.decode(tokens_nuevos, skip_special_tokens=True)


# Pregunta que no está en el dataset de entrenamiento
PREGUNTA_TEST = "¿Qué es un dictionary comprehension en Python?"

print("Respuesta del modelo BASE (antes del fine-tuning):")
print("-" * 60)
respuesta_base = generar_respuesta(PREGUNTA_TEST, modelo, tokenizador)
print(respuesta_base)

Respuesta del modelo BASE (antes del fine-tuning):
------------------------------------------------------------
**Dicionario Comprehension (Dictionary Comprehension)**

Un dictionnaire comprehensión, también conocido como expresiones conjuntivas o listas condicionalmente construidos, es una forma poderosa y concisa para crear diccionarios dinámicos a partir de iterables.

En resumen:

- Un **dictionário**, por sus siglas en inglés `dict`, se define mediante la sintaxis siguiente.
```python
[diccionarion] = [v1 for v2 in iterable]
```
Por ejemplo:


*   Dado el conjunto `{a=10,b,c}` 
    ```python
diccionarion_a_10_b_c = {k:v for k,v in [(x,y)for x,y in [[(i,j),(11,a),12,(13,d)],[(14,e,f)]]]}
print(diccionarion)
# Output :{(0, 'b', c): {'e': f


---
## Parte 6 — Entrenamiento con SFTTrainer

In [31]:
# Con solo 10 ejemplos hacemos pocas épocas para evitar overfitting
# En producción real se necesitan 100+ ejemplos y más épocas

config_entrenamiento = SFTConfig(
    output_dir="./modelo_finetuned",
    num_train_epochs=3,          # épocas: pasadas completas por el dataset
    per_device_train_batch_size=2,  # ejemplos por paso de gradiente
    gradient_accumulation_steps=4,  # acumular gradientes antes de actualizar
    learning_rate=2e-4,          # tasa de aprendizaje: cuánto mover los pesos
    warmup_steps=5,              # pasos iniciales con lr creciente
    logging_steps=5,             # cada cuántos pasos imprimir métricas
    save_strategy="no",          # no guardar checkpoints intermedios
    fp16=False,                  # desactivar — conflicto con cuantización 4-bit
    bf16=False,                  # desactivar — mismo conflicto
    max_length =512,             # longitud máxima de secuencia a tokenizar
    report_to="none",            # no enviar métricas a ningún servicio externo
)

entrenador = SFTTrainer(
    model=modelo,
    args=config_entrenamiento,
    train_dataset=dataset,
    processing_class=tokenizador,
)

print("Configuración del entrenamiento lista")
print(f"Épocas: {config_entrenamiento.num_train_epochs}")
print(f"Batch size efectivo: {config_entrenamiento.per_device_train_batch_size * config_entrenamiento.gradient_accumulation_steps}")
print(f"Learning rate: {config_entrenamiento.learning_rate}")

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Configuración del entrenamiento lista
Épocas: 3
Batch size efectivo: 8
Learning rate: 0.0002


In [32]:
# Lanzar el entrenamiento
# Verás los logs con el valor de loss en cada logging_step
# El loss debería bajar gradualmente — eso indica que el modelo está aprendiendo

print("Iniciando entrenamiento...")
resultado_entrenamiento = entrenador.train()

print(f"\nEntrenamiento completado")
print(f"Loss final: {resultado_entrenamiento.training_loss:.4f}")
print(f"Pasos totales: {resultado_entrenamiento.global_step}")

Iniciando entrenamiento...


Step,Training Loss
5,2.063474



Entrenamiento completado
Loss final: 1.9987
Pasos totales: 6


---
## Parte 7 — Inferencia después del fine-tuning y comparación

In [33]:
# Probar el modelo fine-tuneado con la misma pregunta que usamos como línea base

print("Respuesta del modelo FINE-TUNEADO:")
print("-" * 60)
respuesta_finetuneada = generar_respuesta(PREGUNTA_TEST, modelo, tokenizador)
print(respuesta_finetuneada)

Respuesta del modelo FINE-TUNEADO:
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


**Diccionario Comprehension (Conciso)**

Un diccionario comprehensión, también conocido como `dict.fromkeys()`, permite crear objetos dinámicos que contienen claves y valores asociados utilizando la sintaxis concisa.

```python
d = {key for key in [1, 'a', True] if not isinstance(key, bool)}
```

En resumen:

*   **Sintaxis**: `{clave: valor para cada clave}`
*   **Uso general:** Crear objetos sin necesidad explícita del método `append()` o `insert()
*   Ejemplos comunes:
    *   Cargar datos a partir de listas (`{x for x in lista}`)
    *   Eliminar elementos por condiciones (`{k for k in conjunto si no es boolean}`)
    *   Generador iterativo simple (`(for i in iterable) -> value)`)

Por ejemplo, consideremos el


In [34]:
# Comparación lado a lado
print("COMPARACIÓN: Base vs Fine-tuneado")
print(f"Pregunta: {PREGUNTA_TEST}")
print()
print("MODELO BASE:")
print(respuesta_base)
print()
print("MODELO FINE-TUNEADO:")
print(respuesta_finetuneada)
print()

# verificar si el modelo fine-tuneado siguió el formato esperado
formato_correcto = all(
    tag in respuesta_finetuneada
    for tag in ["[Concepto]", "[Ejemplo]", "[Cuándo usarlo]"]
)
print(f"Modelo fine-tuneado siguió el formato [Concepto]/[Ejemplo]/[Cuándo usarlo]: {formato_correcto}")

COMPARACIÓN: Base vs Fine-tuneado
Pregunta: ¿Qué es un dictionary comprehension en Python?

MODELO BASE:
**Dicionario Comprehension (Dictionary Comprehension)**

Un dictionnaire comprehensión, también conocido como expresiones conjuntivas o listas condicionalmente construidos, es una forma poderosa y concisa para crear diccionarios dinámicos a partir de iterables.

En resumen:

- Un **dictionário**, por sus siglas en inglés `dict`, se define mediante la sintaxis siguiente.
```python
[diccionarion] = [v1 for v2 in iterable]
```
Por ejemplo:


*   Dado el conjunto `{a=10,b,c}` 
    ```python
diccionarion_a_10_b_c = {k:v for k,v in [(x,y)for x,y in [[(i,j),(11,a),12,(13,d)],[(14,e,f)]]]}
print(diccionarion)
# Output :{(0, 'b', c): {'e': f

MODELO FINE-TUNEADO:
**Diccionario Comprehension (Conciso)**

Un diccionario comprehensión, también conocido como `dict.fromkeys()`, permite crear objetos dinámicos que contienen claves y valores asociados utilizando la sintaxis concisa.

```python
d = 

In [35]:
# Probar con más preguntas para ver qué tan bien generalizó el modelo
# Estas preguntas no están en el dataset de entrenamiento

preguntas_test = [
    "¿Qué es el operador * para desempaquetar argumentos en Python?",
    "¿Qué es una lambda en Python?",
    "¿Qué es el método __str__ en una clase Python?",
]

for pregunta in preguntas_test:
    print(f"\nPregunta: {pregunta}")
    print("-" * 50)
    respuesta = generar_respuesta(pregunta, modelo, tokenizador)
    print(respuesta)

    sigue_formato = all(
        tag in respuesta
        for tag in ["[Concepto]", "[Ejemplo]", "[Cuándo usarlo]"]
    )
    print(f"  → Formato correcto: {sigue_formato}")


Pregunta: ¿Qué es el operador * para desempaquetar argumentos en Python?
--------------------------------------------------
Operador `*` (desemparado) permite que se multipliquen los elementos del arreglo o iterable a través de la función multiplicadora (`len()`), lo cual devuelve cuántas veces puede ser aplicada la función.

```python
def factorial(n):
    return len(list(range(1, n + 1))) ** n # imprime el resultado del producto de todos los números enteros desde 0 hasta 'n'

print(factorial(*range(5)))
```

En esta expresión, cuando llamamos al método `factorial`, obtenemos:

- `[1]`: El primer elemento.
- `[2, 4]`: Los dos siguientes elementos.
- `[8, 16, ... , 120]`: Todos los restantes y su producto. 

Este último paso calcula el valor final usando exponente log10(len([1..])^n). Por ejemplo,
   - Para obtener [20], usaremos [*][9].

  → Formato correcto: False

Pregunta: ¿Qué es una lambda en Python?
--------------------------------------------------
Lambda función en Python 
Un

---
## Parte 8 — Guardar el adaptador LoRA

Con LoRA no guardamos todo el modelo. Solo guardamos los adaptadores (unos pocos MB).
Para usar el modelo fine-tuneado se carga el modelo base + los adaptadores.

In [36]:
RUTA_ADAPTADOR = "./adaptador_lora_python"

# guardar solo los pesos del adaptador LoRA, no el modelo completo
modelo.save_pretrained(RUTA_ADAPTADOR)
tokenizador.save_pretrained(RUTA_ADAPTADOR)

# ver el tamaño de lo guardado
import os
archivos = os.listdir(RUTA_ADAPTADOR)
print(f"Archivos guardados en {RUTA_ADAPTADOR}:")
for archivo in archivos:
    ruta = os.path.join(RUTA_ADAPTADOR, archivo)
    size_mb = os.path.getsize(ruta) / (1024 * 1024)
    print(f"  {archivo}: {size_mb:.1f} MB")

print()
print("El adaptador LoRA ocupa solo una fracción del tamaño del modelo completo.")
print("Para cargarlo: AutoModelForCausalLM.from_pretrained(base) + PeftModel.from_pretrained(adaptador)")

Archivos guardados en ./adaptador_lora_python:
  tokenizer.json: 16.4 MB
  tokenizer_config.json: 0.0 MB
  adapter_model.safetensors: 6.5 MB
  README.md: 0.0 MB
  chat_template.jinja: 0.0 MB
  adapter_config.json: 0.0 MB

El adaptador LoRA ocupa solo una fracción del tamaño del modelo completo.
Para cargarlo: AutoModelForCausalLM.from_pretrained(base) + PeftModel.from_pretrained(adaptador)


---
## Parte 9 — Reflexión: ¿cuándo fine-tunear vs prompt engineering?

Esta tabla resume las situaciones de cada enfoque.

In [37]:
# Resumen comparativo basado en lo que hicimos en este laboratorio
print("CUÁNDO USAR CADA ENFOQUE")
print("=" * 60)

comparativa = [
    ("Formato de respuesta específico",  "Difícil de mantener en prompts",  "Ideal"),
    ("Dominio o vocabulario específico", "Funciona con few-shot",           "Mejor a largo plazo"),
    ("Dataset de ejemplos disponible",   "No requiere datos",               "Requiere 50+ ejemplos"),
    ("Costo de inferencia",              "Prompt largo = más tokens",       "Prompt corto post FT"),
    ("Tiempo de implementación",         "Horas",                           "Días"),
    ("GPU requerida",                    "No",                              "Sí"),
    ("Conocimiento cambia frecuente",    "Fácil de actualizar",             "Requiere re-entrenar"),
]

print(f"{'Criterio':<40} {'Prompt Engineering':<22} {'Fine-tuning'}")
print("-" * 80)
for criterio, pe, ft in comparativa:
    print(f"  {criterio:<38} {pe:<22} {ft}")

CUÁNDO USAR CADA ENFOQUE
Criterio                                 Prompt Engineering     Fine-tuning
--------------------------------------------------------------------------------
  Formato de respuesta específico        Difícil de mantener en prompts Ideal
  Dominio o vocabulario específico       Funciona con few-shot  Mejor a largo plazo
  Dataset de ejemplos disponible         No requiere datos      Requiere 50+ ejemplos
  Costo de inferencia                    Prompt largo = más tokens Prompt corto post FT
  Tiempo de implementación               Horas                  Días
  GPU requerida                          No                     Sí
  Conocimiento cambia frecuente          Fácil de actualizar    Requiere re-entrenar


---
## Reflexión del alumno

1. ¿El modelo fine-tuneado siguió el formato [Concepto]/[Ejemplo]/[Cuándo usarlo] en todas las preguntas de test? ¿Para cuáles falló?

*(escribe aquí)*

---

2. Con solo 10 ejemplos de entrenamiento, ¿qué limitaciones observas en el modelo fine-tuneado? ¿Qué harías para mejorarlos?

*(escribe aquí)*

---

3. ¿Por qué solo guardamos los adaptadores LoRA y no el modelo completo? ¿Qué ventaja práctica tiene esto?

*(escribe aquí)*

---
*Laboratorio 03 — Python AI Developer 2026 · IES Cibertec*